In [15]:
from pathlib import Path
import pandas as pd

data_path = Path(r"D:\1000_DataScience_MachineLearning\1000_ML_Projects\Attention_Prototype\data\raw_csv\baywa_joined_tables.csv")

df = pd.read_csv(
    data_path,
    sep=";",
    encoding="cp1252",
    engine="python",
    dtype="str",
    na_filter=False,
)

df.columns.tolist(), df.shape

(['WRBTR_s',
  'MWSKZ_s',
  'XBLNR',
  'WAERS',
  'SGTXT',
  'BKTXT',
  'HKONT',
  'BUDAT_year',
  'BUDAT_month',
  'BUDAT_day',
  'BLDAT_year',
  'BLDAT_month',
  'BLDAT_day'],
 (191372, 13))

In [16]:
target_col = "HKONT"

num_cols = ["WRBTR_s"]

cat_cols = [
    "MWSKZ_s",
    "WAERS",
    "BUDAT_year",
    "BUDAT_month",
    "BUDAT_day",
    "BLDAT_year",
    "BLDAT_month",
    "BLDAT_day",
]

work_df = df[num_cols + cat_cols + [target_col]].copy()

# einfache Bereinigung
for c in num_cols:
    work_df[c] = (
        work_df[c]
        .str.replace(",", ".", regex=False)
        .replace("", "0")
        .astype(float)
    )

for c in cat_cols + [target_col]:
    work_df[c] = work_df[c].fillna("").replace("", "__MISSING__").astype(str)

work_df.shape, work_df[target_col].nunique(), work_df[target_col].value_counts().head(20)

((191372, 10),
 184,
 HKONT
 0059000500    12041
 0062355090    11736
 0062355040    11657
 0063890050    10604
 0062350090    10533
 0062300100     6607
 0062355020     6359
 0062200200     5930
 0062600001     5458
 0062600000     4423
 0062250000     4362
 0062200000     4276
 0062100030     4204
 0062450000     4097
 0062156010     4064
 0062202000     3837
 0063890030     3700
 0062210010     3626
 0062400800     3364
 0062500010     3246
 Name: count, dtype: int64)

In [17]:
TOP_N = 20

top_classes = work_df[target_col].value_counts().head(TOP_N).index.tolist()

proto_df = work_df[work_df[target_col].isin(top_classes)].copy()

proto_df.shape, proto_df[target_col].nunique()

((124124, 10), 20)

In [18]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
proto_df["target"] = le.fit_transform(proto_df[target_col])

n_classes = len(le.classes_)
n_classes

20

In [19]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    proto_df,
    test_size=0.2,
    stratify=proto_df["target"],
    random_state=42,
)

train_df.shape, test_df.shape

((99299, 11), (24825, 11))